# 第 08 天：质量因子 1

> 来自《30 天因子研究计划》第 8 天  
> 主题：质量因子 1  
> 必做：ROE / ROA  
> 选做：毛利率  
> 目标产出：质量因子库

---

## 0. 今天你要真正学会什么？

第 6、7 天我们研究了价值因子：一家公司贵不贵。  
今天开始研究质量因子：一家公司好不好。

质量因子的核心问题是：

> 公司能否用资产和股东资本持续创造高质量利润？

今天重点学习：

1. ROE、ROA、毛利率分别是什么。
2. 为什么 ROE 高不一定代表质量高。
3. 如何识别高杠杆推高 ROE 的情况。
4. 如何构建质量因子库。
5. 如何把多个质量指标合成一个综合质量分数。

一句话版：

> 价值因子问“买得便宜吗”，质量因子问“买到的是不是好生意”。

---

## 1. 先建立直觉：好生意是什么样？

一家好公司通常有几个特征：

- 用较少资产赚较多利润。
- 股东资本回报率高。
- 产品或服务有定价权。
- 成本控制好。
- 利润不是靠一次性项目堆出来。

质量因子就是把这些直觉变成可计算指标。

今天先学三个最基础指标：

| 指标 | 关注点 |
| --- | --- |
| ROE | 股东资本回报 |
| ROA | 总资产回报 |
| 毛利率 | 产品或服务的盈利空间 |

---

## 2. 三个核心质量指标

### 2.1 ROE：净资产收益率

公式：


ROE = 净利润 / 净资产


直觉：

> 股东投入的每一元净资产，公司能赚多少钱。

ROE 高通常代表公司为股东创造利润的能力强。

风险点：

- 高杠杆会推高 ROE。
- 净资产很小会让 ROE 虚高。
- 一次性利润会抬高 ROE。

### 2.2 ROA：总资产收益率

公式：


ROA = 净利润 / 总资产


直觉：

> 公司使用全部资产创造利润的效率。

ROA 相比 ROE 更不容易被杠杆美化。

### 2.3 毛利率

公式：


毛利率 = (营业收入 - 营业成本) / 营业收入


直觉：

> 每卖出 1 元产品，扣掉直接成本后能留下多少钱。

毛利率高可能代表：

- 品牌力强
- 技术壁垒高
- 定价权强
- 成本控制好

风险点：

- 不同行业毛利率不可直接比较。
- 高毛利不一定高净利，费用也很重要。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 构造模拟财务数据


In [ ]:
n = 550

df = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "revenue": rng.lognormal(mean=8.7, sigma=0.8, size=n),
    "gross_margin_true": np.clip(rng.normal(0.34, 0.14, size=n), 0.03, 0.85),
    "total_assets": rng.lognormal(mean=9.3, sigma=0.9, size=n),
    "book_equity": rng.lognormal(mean=8.6, sigma=0.9, size=n),
})

df["gross_profit"] = df["revenue"] * df["gross_margin_true"]
operating_expense_ratio = np.clip(rng.normal(0.20, 0.08, size=n), 0.03, 0.55)
df["net_income"] = df["gross_profit"] - df["revenue"] * operating_expense_ratio

# 制造少量亏损和净资产异常
df.loc[rng.choice(n, size=35, replace=False), "net_income"] *= -0.6
df.loc[rng.choice(n, size=12, replace=False), "book_equity"] *= 0.05

df.head()


---

## 5. 计算 ROE、ROA、毛利率


In [ ]:
df["roe"] = df["net_income"] / df["book_equity"]
df["roa"] = df["net_income"] / df["total_assets"]
df["gross_margin"] = df["gross_profit"] / df["revenue"]

df[["ticker", "roe", "roa", "gross_margin"]].head()


描述统计：


In [ ]:
df[["roe", "roa", "gross_margin"]].describe()


注意：

- ROE 可能极端大，尤其当净资产很小时。
- ROA 通常比 ROE 小。
- 毛利率应该大多在 0 到 1 之间。

---

## 6. 高 ROE 的陷阱：杠杆

ROE 可以拆成：


ROE = ROA × 权益乘数
权益乘数 = 总资产 / 净资产


如果公司大量借债，净资产相对资产很小，ROE 可能被放大。


In [ ]:
df["equity_multiplier"] = df["total_assets"] / df["book_equity"]

roe_leverage_view = df[["ticker", "roe", "roa", "equity_multiplier"]].sort_values(
    "roe",
    ascending=False
).head(10)

roe_leverage_view


如果你看到 ROE 很高但 ROA 一般、权益乘数极高，就要小心：

> 这可能不是经营质量好，而是杠杆很高。

---

## 7. 清洗和标准化质量因子

### 7.1 工具函数


In [ ]:
def winsorize_series(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


### 7.2 构造因子

ROE、ROA、毛利率通常是越高越好。


In [ ]:
df["quality_roe"] = zscore(winsorize_series(df["roe"]))
df["quality_roa"] = zscore(winsorize_series(df["roa"]))
df["quality_gross_margin"] = zscore(winsorize_series(df["gross_margin"]))

quality_cols = ["quality_roe", "quality_roa", "quality_gross_margin"]
df["quality_score"] = df[quality_cols].mean(axis=1)

df[["ticker", "roe", "roa", "gross_margin", "quality_score"]].head()


### 7.3 加一个杠杆惩罚版本

为了避免高杠杆虚高 ROE，我们可以构造一个简化惩罚项：


In [ ]:
df["leverage_penalty"] = zscore(winsorize_series(df["equity_multiplier"]))
df["quality_score_adj"] = df["quality_score"] - 0.25 * df["leverage_penalty"]

df[["ticker", "quality_score", "equity_multiplier", "quality_score_adj"]].head()


这不是唯一正确做法，但能帮助你理解：

> 因子构造不是机械套公式，而是要把财务含义放进去。

---

## 8. 检查质量因子关系


In [ ]:
factor_cols = ["quality_roe", "quality_roa", "quality_gross_margin", "quality_score", "quality_score_adj"]
factor_corr = df[factor_cols].corr()
factor_corr


画相关性热力图：


In [ ]:
plt.imshow(factor_corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(factor_cols)), factor_cols, rotation=45)
plt.yticks(range(len(factor_cols)), factor_cols)
plt.title("Quality Factor Correlation")
plt.tight_layout()
plt.show()


ROE 和 ROA 高度相关时，说明它们信息重叠较多。  
毛利率相关性较低时，可能提供产品盈利能力维度的补充。

---

## 9. 模拟未来收益检验

我们让调整后的质量分数对未来收益有一点正向影响。


In [ ]:
noise = rng.normal(0, 0.055, size=n)
df["future_20d_ret"] = 0.011 * df["quality_score_adj"] + noise

rank_ic = df["quality_score_adj"].corr(df["future_20d_ret"], method="spearman")
print("质量综合分数 Rank IC:", round(rank_ic, 4))


分组检验：


In [ ]:
valid = df.dropna(subset=["quality_score_adj", "future_20d_ret"]).copy()
valid["group"] = pd.qcut(
    valid["quality_score_adj"].rank(method="first"),
    q=5,
    labels=["G1 低质量", "G2", "G3", "G4", "G5 高质量"]
)

group_report = valid.groupby("group", observed=True).agg(
    stock_count=("ticker", "count"),
    avg_quality=("quality_score_adj", "mean"),
    avg_roe=("roe", "mean"),
    avg_roa=("roa", "mean"),
    avg_gross_margin=("gross_margin", "mean"),
    avg_future_20d_ret=("future_20d_ret", "mean"),
)

group_report


画图：


In [ ]:
group_report["avg_future_20d_ret"].plot(kind="bar", title="质量分组未来 20 日平均收益")
plt.ylabel("Future 20D Return")
plt.xticks(rotation=30)
plt.show()


---

## 10. 今日目标产出：质量因子库


In [ ]:
def build_quality_factor_library(raw: pd.DataFrame) -> pd.DataFrame:
    out = raw.copy()

    out["roe"] = out["net_income"] / out["book_equity"]
    out["roa"] = out["net_income"] / out["total_assets"]
    out["gross_margin"] = out["gross_profit"] / out["revenue"]
    out["equity_multiplier"] = out["total_assets"] / out["book_equity"]

    out["quality_roe"] = zscore(winsorize_series(out["roe"]))
    out["quality_roa"] = zscore(winsorize_series(out["roa"]))
    out["quality_gross_margin"] = zscore(winsorize_series(out["gross_margin"]))
    out["leverage_penalty"] = zscore(winsorize_series(out["equity_multiplier"]))

    out["quality_score"] = out[[
        "quality_roe",
        "quality_roa",
        "quality_gross_margin",
    ]].mean(axis=1)
    out["quality_score_adj"] = out["quality_score"] - 0.25 * out["leverage_penalty"]

    return out[[
        "ticker",
        "roe",
        "roa",
        "gross_margin",
        "equity_multiplier",
        "quality_roe",
        "quality_roa",
        "quality_gross_margin",
        "quality_score",
        "quality_score_adj",
    ]]


quality_library = build_quality_factor_library(df)
quality_library.head()


这就是今天的目标产出：质量因子库。

---

## 11. 实战注意事项

### 11.1 行业差异

毛利率行业差异巨大。软件公司和超市公司不能直接比较毛利率。

### 11.2 ROE 的可持续性

一年高 ROE 不代表长期高质量。后面可以看多年均值和稳定性。

### 11.3 杠杆影响

高 ROE 可能来自高负债。ROA、权益乘数能帮助识别。

### 11.4 利润质量

净利润不等于现金流。第 9 天会学习现金流质量。

---

## 12. 今天的知识图谱


In [ ]:
mindmap
  root((质量因子1))
    ROE
      净利润除以净资产
      股东资本回报
      高杠杆会放大
      净资产过小会异常
    ROA
      净利润除以总资产
      资产使用效率
      比ROE更少受杠杆影响
    毛利率
      毛利除以收入
      定价权
      成本控制
      行业差异大
    处理
      去极值
      标准化
      杠杆惩罚
      综合打分
    输出
      quality_roe
      quality_roa
      quality_gross_margin
      quality_score


---

## 13. 初学者最容易踩的 7 个坑

### 坑 1：只看 ROE

ROE 高可能来自高杠杆，不一定代表经营质量好。

### 坑 2：忽略 ROA

ROA 能帮助你看总资产使用效率。

### 坑 3：跨行业比较毛利率

行业商业模式不同，毛利率中枢差异很大。

### 坑 4：不处理极端 ROE

净资产很小时，ROE 会非常夸张。

### 坑 5：把一次性利润当质量

一次性收益抬高净利润，会污染 ROE 和 ROA。

### 坑 6：忽略亏损公司

亏损公司质量指标可能为负，需要单独解释。

### 坑 7：质量好不等于价格合适

好公司太贵，也未必是好投资。

---

## 14. 今天的动手作业

### 作业 A：解释指标

用自己的话解释：

1. ROE 是什么？
2. ROA 是什么？
3. 毛利率说明什么？

### 作业 B：运行质量因子库

运行本文代码，输出：

- `quality_roe`
- `quality_roa`
- `quality_gross_margin`
- `quality_score_adj`

### 作业 C：检查高 ROE 股票

查看 ROE 最高的 10 只股票，判断它们是否也有高 ROA。

### 作业 D：调整杠杆惩罚

把惩罚系数从 `0.25` 改成：


In [ ]:
0
0.5
1.0


观察高质量组如何变化。

---

## 15. 自测题

### 题 1

ROE 的公式是什么？

答案：净利润 / 净资产。

### 题 2

ROA 的公式是什么？

答案：净利润 / 总资产。

### 题 3

为什么高 ROE 不一定好？

答案：高 ROE 可能来自高杠杆、净资产过小或一次性利润。

### 题 4

毛利率高通常说明什么？

答案：可能说明产品定价权强、成本控制好或商业模式较好。

### 题 5

为什么质量因子也要做去极值？

答案：财务比率容易出现极端值，可能扭曲标准化和排序。

---

## 16. 今日复盘模板


第 08 天复盘：质量因子 1

1. 我今天理解的 ROE：

2. 我今天理解的 ROA：

3. 我今天理解的毛利率：

4. 我构建的质量因子字段：

5. 我观察到的高 ROE 陷阱：

6. 我认为质量因子最大风险：

7. 明天学习净利润增长率和现金流质量前，我需要准备：


---

## 17. 明天预告：质量因子 2

明天会继续质量因子，但从静态盈利能力走向动态和现金流：


净利润增长率 + 现金流质量


这会帮助你判断利润是不是在增长，以及利润有没有现金流支撑。

---

## 18. 一句话收尾

质量因子不是寻找“看起来利润高”的公司，而是寻找真正能持续、高效、健康赚钱的公司。

> 高 ROE 很迷人，但能被 ROA、毛利率和现金流一起支持的高 ROE 才更值得信任。

---

## 19. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释质量因子的计算方法，不构成任何投资建议。真实研究需要处理财务披露时点、行业差异、会计口径、一次性损益、杠杆影响和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 08 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：质量因子1
- 必做：ROE/ROA
- 选做：毛利率
- 目标产出：质量因子库

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

高 ROE 很迷人，但如果靠高杠杆堆出来，风险也在同步放大。质量因子要看盈利能力，也要看资产效率和商业模式。

这个例子背后的关键直觉是：

> 好公司不是利润高一次，而是高效、稳健、可持续地赚钱。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


质量因子1
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 质量因子库


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(108)
n = 350
df = pd.DataFrame({
    "ticker": [f"S{i:03d}" for i in range(n)],
    "revenue": rng.lognormal(8.5, .8, n),
    "assets": rng.lognormal(9.2, .8, n),
    "equity": rng.lognormal(8.4, .8, n),
})
margin = np.clip(rng.normal(.35, .12, n), .03, .8)
df["gross_profit"] = df["revenue"] * margin
df["net_income"] = df["gross_profit"] - df["revenue"] * np.clip(rng.normal(.22, .08, n), .03, .55)
df["roe"] = df["net_income"] / df["equity"]
df["roa"] = df["net_income"] / df["assets"]
df["gross_margin"] = df["gross_profit"] / df["revenue"]

def zscore(s):
    s = s.clip(s.quantile(.01), s.quantile(.99))
    return (s - s.mean()) / s.std()

df["quality_score"] = pd.concat([zscore(df["roe"]), zscore(df["roa"]), zscore(df["gross_margin"])], axis=1).mean(axis=1)
print(df[["roe", "roa", "gross_margin", "quality_score"]].describe().round(3))


## E. 产出验收标准

完成今天课程后，你的 `质量因子库` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `质量因子1` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `质量因子库` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `质量因子库`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 08 天复盘：质量因子1

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
